In [ ]:
import os
import warnings

# Note we cannot use rasterio *raw* with matplot/folium. The memory req. to load the moderate size datasets are off the chart
# Also impacts on using plotting off this library (mem manage?)
try:
    import rasterio
    import rasterio.plot
    from rasterio.plot import show_hist
except:
    !pip install rasterio

# Local streaming really the only solution here
try:
    from localtileserver import get_folium_tile_layer, TileClient
except:
    !pip install localtileserver
try:
    import folium
except:
    !pip install folium
try:
    import geopandas
except:
    !pip install geopandas

In [ ]:
datapath = "data/313f4008-3930-4dea-ba0f-aac448fb3e8c/"
filename = "ID106_N60_W10_RP100_depth.tif"
datapath1 = os.path.join(datapath, filename)

filename = "ID113_N60_W0_RP100_depth.tif"
datapath2 = os.path.join(datapath, filename)

In [ ]:
# Load in the (floodmap) geotiffs as rasters for analysis (do NOT plot objs directly - it's inefficient)
robj1 = rasterio.open(datapath1)
robj2 = rasterio.open(datapath2)

robj1.meta

In [ ]:
# We need to load in the regions map as well so we can effectively use a subset of the historical site data
mdatapath = "data/e25a5e5d-2eb7-4c77-a5f7-919ee35a863a"
hjnfile = "Counties_and_Unitary_Authorities_December_2024_Boundaries_UK_BGC_3152178837812104842.geojson"
hjnfile = os.path.join(mdatapath, hjnfile)

uk_regions = geopandas.read_file(hjnfile)
uk_regions.head()

In [ ]:
# Map
mreg = folium.Map([51, 0.0], zoom_start=4)
folium.GeoJson(
    uk_regions,
    name="UK Regions",
    tooltip=folium.GeoJsonTooltip(fields=["CTYUA24NM"]),
    fill_opacity=0.1,
    line_weight=0.1,
).add_to(mreg)
mreg

In [ ]:
# Select only data that is in the specified region - from above map - (we want the region_geom geometry/borders)
select_region = 'Wiltshire'
reg_row = uk_regions.loc[uk_regions['CTYUA24NM'] == select_region]
# Now get the polygon from the row (should just be one)
region_geom = reg_row['geometry'].iloc[0]

#print(testg, type(testg))
reg_row.head()

In [ ]:
# Load in the Historic England Listed buildings & then check what's there.
mdatapath = "data/e6435368-ec62-43b9-a1be-6133828fa10b"
hjnfile = "National_Heritage_List_for_England_NHLE_v02_VIEW_8753755809632832301-ListedBuildings.geojson"

hjnfile = os.path.join(mdatapath, hjnfile)

gdfh1 = geopandas.read_file(hjnfile)
gdfh1.head()

In [ ]:
print(gdfh1.dtypes)

In [ ]:
# Warning that England full file is way too big for doing everything in one go!

# Some data cleaning on the main dataframe & 
# Create active hyperlinks
gdfh1['href'] = '<a href="' + gdfh1.hyperlink + '">' + gdfh1.hyperlink + "</a>"
gdfh1 = gdfh1.explode(index_parts=False)  # This btw fixes the multipoint issue (multipoint introduces a 'click on data' bug)

# Select elect points based on region selected above
gdfh1_reg = gdfh1[gdfh1.geometry.within(region_geom)]

# Also CREATE mini-test version with only 100 entries (listed sites)
gdfh1t = gdfh1.iloc[0:100]

# Quick look at data to be sure it makes sense.
gdfh1_reg.head()

In [ ]:
print(gdfh1_reg.dtypes)

In [ ]:
# NOT needed now but keep here for example operations.
#gdfh1t.rename(columns={'OBJECTID': 'objectid'}, inplace=True)
#gdfh1t['objectid'] = gdfh1t['objectid'].astype('str')
#gdfh1t = gdfh1t.explode(index_parts=False) # This fixes the multipoint issue (multipoint introduces a 'click on data' bug)
#gdfh1t.head()
#print(gdfh1t.dtypes)

In [ ]:
#lat, lon = 51.4042789,-0.3466018  # 51.4035436,-0.3375485
#inund1 = list(robj1.sample([(lon, lat)]))[0]
#inund2 = list(robj2.sample([(lon, lat)]))[0]
# print(f"Chk : Flood extent (either {inund1[0]},{inund2[0]}). Pick max:", max(inund1[0], inund2[0]), type(inund1[0]))

# Now add an extra column for FLOOD innundation data
# gdfh1_reg, get the site (data) Lats & Lons from the geometry for testing
gdfh1_reg['lon'] = gdfh1_reg.geometry.x
gdfh1_reg['lat'] = gdfh1_reg.geometry.y

def get_flooding(lon, lat):
    inund1 = list(robj1.sample([(lon, lat)]))[0]
    inund2 = list(robj2.sample([(lon, lat)]))[0]
    max_inund = max(inund1[0], inund2[0], 0.0)
    # print("Coords", lat, lon, "Flooding" ,inund1, inund2, max_inund)
    return max_inund

# Create flood level column
flood_lvl = [get_flooding(x, y) for x, y in zip(gdfh1_reg['lon'], gdfh1_reg['lat'])]
# Add new column
gdfh1_reg['floodlevel'] = flood_lvl


# Now create a column
def fc(x):
    if x > 10.0: return 4
    if x > 3.0: return 3
    if x > 1.0: return 2
    if x > 0.1: return 1
    else: return 0
colors = ["green", "yellow", "orange", "red", "purple"]

gdfh1_reg['floodclass'] = gdfh1_reg['floodlevel'].apply(lambda x: fc(x))
gdfh1_reg

In [ ]:
flood_classes = gdfh1_reg.floodclass.unique().tolist()
print(flood_classes)
gdfh1_reg.dtypes

In [ ]:
warnings.filterwarnings('ignore') # Supress the numpy warnings, hundreds+ of lines.

client1 = TileClient(datapath1)
client2 = TileClient(datapath2)
mapobj = folium.Map(location=client1.center())

t1 = get_folium_tile_layer(client1, opacity=0.50)
t2 = get_folium_tile_layer(client2, opacity=0.50)
t1.add_to(mapobj)
#t2.add_to(mapobj)

# Add the historical data (& hope folium does the layers correctly in order)
folium.GeoJson(
    gdfh1_reg,
    name="Historical Sites",
    marker=folium.Circle(radius=20, fill_color="green", fill_opacity=0.4, color="black", weight=1),
    tooltip=folium.GeoJsonTooltip(fields=["Name", "Grade", "floodlevel", "floodclass"]),
    popup=folium.GeoJsonPopup(fields=["Name", "Grade", "href", "floodlevel", "floodclass"]),
    style_function=lambda x: {
        "fillColor": colors[x['properties']['floodclass']],
    },
    highlight_function=lambda x: {"fillOpacity": 0.8},
    zoom_on_click=True,
).add_to(mapobj)

mapobj